In [1]:
import dspy
import os
from dotenv import load_dotenv

load_dotenv()

ANTHROPIC_KEY = os.getenv("ANTHROPIC_KEY")
OPENAI_KEY = os.getenv("OPENAI_KEY")

In [2]:
dspy.__version__

'3.1.3'

In [3]:
lm = dspy.LM("anthropic/claude-opus-4-6", api_key=ANTHROPIC_KEY)
dspy.configure(lm=lm)

In [4]:
#Direct API call

simple_article = lm("Create a simple article about travelling in Canada")[0]
print(simple_article[:1000])

# Travelling in Canada: A Land of Endless Adventures

Canada, the second-largest country in the world, is a dream destination for travellers seeking breathtaking landscapes, vibrant cities, and unforgettable experiences. From coast to coast, this diverse nation offers something for everyone, no matter what kind of adventure you're looking for.

## Natural Wonders

Canada is home to some of the most stunning natural scenery on the planet. The Rocky Mountains in Alberta and British Columbia offer towering peaks, crystal-clear lakes, and world-class hiking trails. Banff and Jasper National Parks attract millions of visitors each year who come to marvel at turquoise waters like Lake Louise and enjoy wildlife spotting along scenic drives.

On the east coast, the rugged shores of Nova Scotia and Newfoundland provide dramatic ocean views, charming fishing villages, and the chance to spot whales and icebergs. Meanwhile, Niagara Falls remains one of the most iconic natural attractions in the wo

# Modules
Module is simply a wrapper around a LLM call, its just that it is more organized and self-explanatory as opposed to simply dumping a whole bunch of instructions prior to LLM call.

Some of the examples are 

1) Predict(which is as simple as calling a LLM)
2) ChainOfThought()
3) ReAct()
4) RAG()
5) ProgramOfThought() -> This is for code generation


# Signatures 
These are simply ways in which you can you can define input/output for a specific module that you are working with.


#### Method 1 - shorthand signature

In [5]:

## M.O -> "question -> answer",  "context, question -> answer", "question -> answer ".
#Additionally  you can also do type checks


#Single i/p, o/p
qa = dspy.Predict("question:str -> answer:str")
print(f"**Single i/p-o/p** \n {qa(question="What is the capital city of Canada?")}")

#Single i/p, multiple o/ps
qa = dspy.Predict("question -> answer1, answer2")
print(f"**Multiple o/p** \n {qa(question="What are primary languages spoken in Canada?")}")


# Also single ip and multiple ops with summarization - This actually kinda cool, given that
#you dont necessarily have to keep telling it what info you need from the LLM call.
#Additionally, this looks more pythonic
qa = dspy.Predict("question:str -> summary:str, keywords:str, description:str")
print(qa(question=f"Give me the requested information for this article{simple_article}"))

##If you want to extract these fields from Prediction object use - result.<arg_name>

**Single i/p-o/p** 
 Prediction(
    answer='The capital city of Canada is Ottawa.'
)
**Multiple o/p** 
 Prediction(
    answer1='English',
    answer2='French'
)
Prediction(
    summary="This article is a comprehensive travel guide to Canada, highlighting the country's natural wonders (Rocky Mountains, Niagara Falls, Atlantic coastlines), vibrant cities (Toronto, Vancouver, Montreal, Quebec City, Ottawa), outdoor activities (skiing, canoeing, scenic drives), and rich cultural experiences (Indigenous heritage, festivals, diverse cuisine). It also provides practical tips for travellers regarding the best times to visit, transportation, and packing advice, concluding that Canada offers an unparalleled travel experience for all types of adventurers.",
    keywords='Canada travel, Canadian adventure, Rocky Mountains, Banff, Jasper, Lake Louise, Niagara Falls, Toronto, Vancouver, Montreal, Quebec City, Ottawa, Nova Scotia, Newfoundland, outdoor activities, skiing, snowboarding, canoeing, Ca

In [6]:
##Debugging the calls to see the actual prompt -> You needs to do this after you are done
#running any of these for once.

#n = number ps previous runs to refer from
dspy.inspect_history(n=3)





[2026-03-22T09:37:52.130661]

System message:

Your input fields are:
1. `question` (str):
Your output fields are:
1. `answer` (str):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## question ## ]]
{question}

[[ ## answer ## ]]
{answer}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Given the fields `question`, produce the fields `answer`.


User message:

[[ ## question ## ]]
What is the capital city of Canada?

Respond with the corresponding output fields, starting with the field `[[ ## answer ## ]]`, and then ending with the marker for `[[ ## completed ## ]]`.


Response:

[[ ## answer ## ]]
The capital city of Canada is Ottawa.

[[ ## completed ## ]]





[2026-03-22T09:37:52.146704]

System message:

Your input fields are:
1. `question` (str):
Your output fields are:
1. `answer1` (str): 
2. `answer2` (str):
All interactions will be structured in the following way, with the appropriate va

#### Method 2 - Class-based - adds descriptions and types
It lets you declare and describe the arguments more systematically. 

There are 2 kinds - Names are self-explanatory
1) InputField
2) OutputField

In [7]:
class SimpleQA(dspy.Signature):
    """Answer questions with short factual responses""" 

    question:str = dspy.InputField(desc="A hypothetical/factual/fictional question")
    answer:str = dspy.OutputField(desc="A factual answer")


sqa = dspy.Predict(signature=SimpleQA)

sqa(question="What is the purpose of life and who should seek it?")

#Note the docstring at the begining will be used by LLM for context. This is survived
#during optimzations

Prediction(
    answer='The purpose of life is a deeply philosophical question with varied answers depending on perspective. Biologically, it is survival and reproduction. Philosophically, thinkers like Aristotle said it is the pursuit of happiness (eudaimonia), while existentialists like Sartre argued each individual must create their own meaning. Religiously, many traditions hold it is to serve God or achieve spiritual liberation. Everyone should seek their own sense of purpose, as the search itself is a universal human endeavor.'
)

In [8]:
reviews = [
    "Absolutely stunning hotel! The staff went above and beyond to make our stay memorable. Room was spotless and the view was breathtaking.",
    "Terrible experience from start to finish. The room smelled musty, the AC was broken, and front desk staff were completely unhelpful.",
    "Decent place for the price. Nothing special, but the bed was comfortable and the location was convenient for the conference I attended.",
    "The spa and pool area were divine, but our room hadn't been properly cleaned — found someone's hair on the pillow. Very disappointing given the price.",
    "Perfect anniversary getaway! Complimentary champagne, rose petals on the bed, and the most attentive service I've ever experienced.",
    "Waited 45 minutes to check in, elevator was out of service the entire stay, and the breakfast was overpriced for what it was.",
    "Solid business hotel. Fast WiFi, good desk setup, and the gym was well-equipped. Won't blow you away but gets the job done.",
    "I've stayed in hostels with better service. Staff were rude, room was tiny, and they charged us for amenities we never used.",
    "Lovely boutique hotel with charming decor and a fantastic rooftop bar. The noise from the street made sleeping difficult though.",
    "Outstanding in every way — immaculate rooms, gourmet breakfast, and a concierge who somehow got us last-minute dinner reservations. Will absolutely return."
]

from typing import Literal ## These are hints not checks

class SentimentClassifier(dspy.Signature):
    """Given a review, assign either a positive, negative or mixed review. 
    DO NOT assign multiple labels."""

    review:str = dspy.InputField(desc="A hotel review describing guest experience with their stay")

    #Type hinting that output can be positive, negative or mixed
    sentiment:Literal["positive", "negative", "mixed"] = dspy.OutputField(desc="Carefully analyze the review and assign either a positive, negative or mixed review")


sentiment_classifier = dspy.Predict(signature=SentimentClassifier)

sentiment_labels = [sentiment_classifier(review=r).sentiment for r in reviews]
print(sentiment_labels)


['positive', 'negative', 'positive', 'mixed', 'positive', 'negative', 'positive', 'negative', 'mixed', 'positive']


In [9]:
##The only way to actually "check" - Note there are fancy ways of doing this within dspy.
#But simpler the better

for label in sentiment_labels:
    assert(label.lower() in ["positive", "negative", "mixed"])
    

In [10]:
dspy.inspect_history(n=1)





[2026-03-22T09:37:52.378502]

System message:

Your input fields are:
1. `review` (str): A hotel review describing guest experience with their stay
Your output fields are:
1. `sentiment` (Literal['positive', 'negative', 'mixed']): Carefully analyze the review and assign either a positive, negative or mixed review
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## review ## ]]
{review}

[[ ## sentiment ## ]]
{sentiment}        # note: the value you produce must exactly match (no extra characters) one of: positive; negative; mixed

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Given a review, assign either a positive, negative or mixed review. 
        DO NOT assign multiple labels.


User message:

[[ ## review ## ]]
Outstanding in every way — immaculate rooms, gourmet breakfast, and a concierge who somehow got us last-minute dinner reservations. Will absolutely return.

Respond with the correspo

#### A simple data extraction problem

In [11]:
bug_reports = [
    "Login page crashes on Safari 17. Steps: 1) Open safari 2) Go to login page 3) Click 'Sign in with Google' 4) Page goes white and console shows TypeError. This blocks all Safari users from logging in. Started after last Friday's deploy.",
    "The export CSV button sometimes produces empty files. Open any report, click export, and about 1 in 5 times the file is 0 bytes. Not urgent since retry works, but annoying. Chrome and Firefox both affected.",
    "Dashboard charts show wrong timezone for EU users. Go to Analytics > Daily view, check the x-axis timestamps. They show PST instead of the user's local time. Cosmetic issue but confusing for our London team.",
    "CRITICAL: Production database connection pool exhausted. App returns 500 errors under load. Steps: hit /api/search with 50 concurrent requests, watch connection count climb, never releases. Started after upgrading to postgres driver v4.2. Rolling back driver fixes it. Need immediate fix, affecting all customers.",
    "Notification emails have broken formatting in Outlook. Open any notification email in Outlook desktop. Images are oversized and the footer overlaps the body text. Gmail and Apple Mail render fine. Low priority cosmetic bug.",
]

In [12]:
dspy.context(lm=lm,
            track_usage=True,
             adapter=dspy.JSONAdapter()
            )

In [13]:
class BugReportExtract(dspy.Signature):
    """ Extract key details from the given bug report. The key details include title,
        severity, and steps to potentially reproduce the bug
    """
    report: str = dspy.InputField(desc="A brief bug report.")
    title: str = dspy.OutputField(desc="A short title to bug report")
    severity:str = dspy.OutputField(desc="One of crtitical, high, medium or low. Should always be one of these")
    steps: list[str] = dspy.OutputField(desc="Extract the steps to reproduce the bug")

bg_report = dspy.Predict(BugReportExtract)

results = [bg_report(report=report) for report in bug_reports]
results
    

[Prediction(
     title="Login page crashes on Safari 17 when clicking 'Sign in with Google'",
     severity='critical',
     steps=['Open Safari 17', 'Go to login page', "Click 'Sign in with Google'", 'Page goes white and console shows TypeError']
 ),
 Prediction(
     title='Export CSV button intermittently produces empty (0 byte) files',
     severity='low',
     steps=['Open any report', 'Click the export CSV button', 'Check the downloaded file size', 'Repeat multiple times — approximately 1 in 5 attempts will produce a 0-byte file', 'Observe the issue occurs in both Chrome and Firefox']
 ),
 Prediction(
     title="Dashboard charts display PST timezone instead of user's local timezone for EU users",
     severity='low',
     steps=['Go to Analytics section', 'Select Daily view', 'Check the x-axis timestamps on the dashboard charts', "Observe that timestamps show PST instead of the user's local timezone (e.g., GMT/BST for London users)"]
 ),
 Prediction(
     title='Production data

In [14]:
dspy.inspect_history(n=1)





[2026-03-22T09:37:52.525361]

System message:

Your input fields are:
1. `report` (str): A brief bug report.
Your output fields are:
1. `title` (str): A short title to bug report
2. `severity` (str): One of crtitical, high, medium or low. Should always be one of these
3. `steps` (list[str]): Extract the steps to reproduce the bug
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## report ## ]]
{report}

[[ ## title ## ]]
{title}

[[ ## severity ## ]]
{severity}

[[ ## steps ## ]]
{steps}        # note: the value you produce must adhere to the JSON schema: {"type": "array", "items": {"type": "string"}}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Extract key details from the given bug report. The key details include title,
        severity, and steps to potentially reproduce the bug


User message:

[[ ## report ## ]]
Notification emails have broken formatting in Outlook. Open any notification e

## ChainofThought
This is another module where instea dof you specifying - carefully think this throught and
reason with this prompt, dspy handles it automatically. reasoning param is enabled automatically

#### Key insights from below
1) Notice how signatures didnt change though the modules were entirely different
2) Reasoning is enabled by default i.e. a programmatic parameter to get reasoning done
3) Notice how even with Predict the result was similar especially when message had 2 possible
departments to route to. It's important to understand that the models are getting better day by day and probbaly cot adds extra tokens and latency. But its is good for auditability purposes - if you need to know why a model made a certain prediction

In [15]:
class TicketRouter(dspy.Signature):
    """ Given a support ticket route it to the right department"""

    ticket:str = dspy.InputField(desc="customer's message")
    category:Literal["billing", "technical", "account", "shipping"] = dspy.OutputField(desc="The only departments a ticket shouule be routed to")

tr_pred = dspy.Predict(TicketRouter)
print(f"**Using Pred** - {tr_pred(ticket="I was charged twice but also cant log in")}")

tr_cot = dspy.ChainOfThought(TicketRouter)
print(f"**Using COT** - {tr_cot(ticket="I was charged twice but also cant log in")}")



**Using Pred** - Prediction(
    category='billing'
)
**Using COT** - Prediction(
    reasoning="The customer has two issues: being charged twice (a billing issue) and being unable to log in (an account issue). The double charge is a financial concern that likely needs immediate attention, and billing issues typically take priority since they involve the customer's money. However, the login issue could also be related to the account being in a problematic state due to the billing issue. Since the double charge is the primary concern and is listed first, routing to billing makes the most sense as they can address the financial issue and potentially escalate the login issue to the account team.",
    category='billing'
)


In [16]:
tr_cot.inspect_history(n=1)





[2026-03-22T09:37:52.610662]

System message:

Your input fields are:
1. `ticket` (str): customer's message
Your output fields are:
1. `reasoning` (str): 
2. `category` (Literal['billing', 'technical', 'account', 'shipping']): The only departments a ticket shouule be routed to
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## ticket ## ]]
{ticket}

[[ ## reasoning ## ]]
{reasoning}

[[ ## category ## ]]
{category}        # note: the value you produce must exactly match (no extra characters) one of: billing; technical; account; shipping

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Given a support ticket route it to the right department


User message:

[[ ## ticket ## ]]
I was charged twice but also cant log in

Respond with the corresponding output fields, starting with the field `[[ ## reasoning ## ]]`, then `[[ ## category ## ]]` (must be formatted as a valid Python Literal['billing', 'tech

In [17]:
## Simple what is wrong with this code

code1 = """
def calculate_average(numbers):
    " Calculate average of numbers in a given list"
    total = 1
    for n in numbers:
        total += n
    return total / len(numbers)
"""

code2 = """
def calculate_average(numbers):
    "Calculate average of numbers in a given list "
    if len(numbers) == 0:
        raise ValueError("List cannot be empty")
    total = 0
    for n in numbers:
        total += n
    return total / len(numbers)

"""



dspy.context(lm=lm,
             adapter=dspy.JSONAdapter)

class CodeReviewer(dspy.Signature):
    """ Review the provided code snippet, explain is it correct or wrong. Either ways
    provide an explaination and if the code is wrong/buggy provide a fix for the same.
    NOTE:- Efficiency is out of scope for this review.
    """

    code:str = dspy.InputField(desc="Code snippet to review")
    verdict:Literal["right", "buggy"] = dspy.OutputField(desc="Classify the code into one of these labels")
    explaination:str = dspy.OutputField(desc="Reasoning as to why the code is either wrong or buggy")
    fix:str = dspy.OutputField(desc="Provide a fix if the code is buggy. Otheriwse simply return an empty string")



code_reviewer = dspy.ChainOfThought(CodeReviewer)
print(f"*** Code 1 verdict *** {code_reviewer(code=code1)}")
print(f"\n*** Code 2 verdict *** {code_reviewer(code=code2)}")




*** Code 1 verdict *** Prediction(
    reasoning='The function is supposed to calculate the average of a list of numbers. The average is defined as the sum of all numbers divided by the count of numbers. Looking at the code, `total` is initialized to `1` instead of `0`. This means every calculated average will be off by `1/len(numbers)` because an extra `1` is being added to the sum.',
    verdict='buggy',
    explaination='The variable `total` is initialized to `1` instead of `0`. This causes the sum to be incorrect (it will always be 1 more than the actual sum), which in turn makes the calculated average incorrect. For example, `calculate_average([2, 4])` would return `(1 + 2 + 4) / 2 = 3.5` instead of the correct answer `3.0`.',
    fix='def calculate_average(numbers):\n    " Calculate average of numbers in a given list"\n    total = 0\n    for n in numbers:\n        total += n\n    return total / len(numbers)'
)

*** Code 2 verdict *** Prediction(
    reasoning="Let me analyze this

## Evaluation

One of the aspects where DSPy stops being a wrapper - NGL; being a wrapper and given how
easy it is organize various aspects of your prompt .

Honestly, you dont needs DSPy to do this you can still wing it with typical prompting/evaluating

1) Create a dspy.Example object - Its a way of hinting that this is a part of some train/eval set
2) Quite easy to use and to also customize
3) 

In [18]:
emails = [
    {"text": "My invoice shows a charge I didn't authorize", "department": "billing", "urgency": "high"},
    {"text": "How do I change my password?", "department": "account", "urgency": "low"},
    {"text": "The app crashes every time I open the dashboard", "department": "technical", "urgency": "high"},
    {"text": "Can you send me a copy of my receipt from last month?", "department": "billing", "urgency": "low"},
    {"text": "Someone logged into my account from another country", "department": "account", "urgency": "high"},
    {"text": "Your website is down and we can't process orders", "department": "technical", "urgency": "high"},
    {"text": "I'd like to upgrade my subscription plan", "department": "billing", "urgency": "low"},
    {"text": "The search feature returns no results for any query", "department": "technical", "urgency": "medium"},
    {"text": "Please delete my account and all associated data", "department": "account", "urgency": "medium"},
    {"text": "I was promised a refund 3 weeks ago and still haven't received it", "department": "billing", "urgency": "high"},
]

In [19]:
dspy.context(lm=lm,cache=False)

# eval_set = [dspy.Example(text=email["text"], department=email["department"], \
#                         urgency=email["urgency"]).with_inputs("text") for email in emails]

#Use dspy.Example? - You can see you can create examples directly from dictionary

eval_set = [dspy.Example(email).with_inputs("text") for email in emails]

class TriageEmail(dspy.Signature):
    """ Classify a customer email by department and urgency."""

    text:str = dspy.InputField(desc="Customer email")
    department:Literal["billing", "account", "technical"] = dspy.OutputField(desc="Classify the email into one of these categories")
    urgency:Literal["high", "medium", "low"] = dspy.OutputField(desc="Classify the email into one of the caetgories of urgency. Consider both business and reputation risk when doing so")

#For brevity lets say both department/urgency should match. No partial credits for now
def triage_metric(example, prediction):

    return (example.department.lower() == prediction.department.lower()) and ( \
        example.urgency.lower() == prediction.urgency.lower())



email_classifier = dspy.ChainOfThought(TriageEmail)

#dspy.Evaluate?

evaluator = dspy.Evaluate(
    devset=eval_set,
    metric=triage_metric,
    display_progress=True,
    provide_traceback=True, save_as_json="./results.json"
)

score = evaluator(email_classifier)
print(f"Score = {score}")




Average Metric: 7.00 / 10 (70.0%): 100%|██████████████████| 10/10 [00:00<00:00, 613.15it/s]

2026/03/22 09:37:52 INFO dspy.evaluate.evaluate: Average Metric: 7 / 10 (70.0%)



Score = EvaluationResult(score=70.0, results=<list of 10 results>)


In [20]:
##Simple Resume screening
applications = [
    {"resume": "5 years Python, led team of 8, built ML pipeline at Series B startup, MS in CS from Stanford", "role_fit": "strong", "experience_level": "senior"},
    {"resume": "Recent bootcamp grad, 2 personal projects in React, no professional experience", "role_fit": "weak", "experience_level": "junior"},
    {"resume": "3 years as data analyst, some Python scripting, transitioning into engineering, BS in Math", "role_fit": "moderate", "experience_level": "mid"},
    {"resume": "10 years Java enterprise, architect at Fortune 500, no Python or ML experience", "role_fit": "moderate", "experience_level": "senior"},
    {"resume": "1 year internship at Google, contributed to open source LLM project, currently finishing PhD in NLP", "role_fit": "strong", "experience_level": "mid"},
    {"resume": "Self-taught developer, 6 months freelancing, built CRUD apps in Django", "role_fit": "weak", "experience_level": "junior"},
    {"resume": "7 years full-stack, 2 years leading platform team, built real-time data pipelines, strong Python", "role_fit": "strong", "experience_level": "senior"},
    {"resume": "2 years QA testing, learning Python on the side, no development role yet", "role_fit": "weak", "experience_level": "junior"},
]

screening_evalset = [dspy.Example(app).with_inputs("resume") for app in applications]
#Create a signature
class ResumeScreen(dspy.Signature):
    """Role:-ML Engineer: Build and deploy machine learning models and data pipelines.
        Requires strong Python, ML frameworks (PyTorch/TensorFlow), 
        and 3+ years experience. Nice to have: cloud platforms, team leadership, NLP."""

    resume:str = dspy.InputField(desc="A brief description of candidate's resume")
    
    role_fit:Literal["strong", "moderate", "weak"] = dspy.OutputField(desc="Assess the resume to check \
    the role fit of the candidate. Ignore any information like pronouns, names, country of origin")
    
    experience_level:Literal["senior","mid","junior"] = dspy.OutputField(desc="Categorize the candidate into one of these buckets. Consider both academic and professional experience")


#Full match for brevity - dont miss trace  especially when using trace
def screen_evaluator(example, prediction, trace=None):

    return (example.role_fit.lower() == prediction.role_fit.lower()) and \
        (example.experience_level.lower() == prediction.experience_level.lower())


pred_screener = dspy.Predict(ResumeScreen)
cot_screener = dspy.ChainOfThought(ResumeScreen)

##Evaluator 
evaluator = dspy.Evaluate(
    devset=screening_evalset,
    metric=screen_evaluator,
    display_progress=True)

evaluator(pred_screener), evaluator(cot_screener) ##Didnt make a huge difference

##1) 62% is your base accuracy. You can try and see if the "optimizer" can improve your score
##2) There is no difference between the 2 in terms of performance as both are using
#claude-opus which is already a reasoning and a very capable model. Using it against a 
#less-efficient model could ne helpful

Average Metric: 5.00 / 8 (62.5%): 100%|█████████████████████| 8/8 [00:00<00:00, 328.14it/s]

2026/03/22 09:37:52 INFO dspy.evaluate.evaluate: Average Metric: 5 / 8 (62.5%)



Average Metric: 5.00 / 8 (62.5%): 100%|█████████████████████| 8/8 [00:00<00:00, 323.60it/s]

2026/03/22 09:37:52 INFO dspy.evaluate.evaluate: Average Metric: 5 / 8 (62.5%)


(EvaluationResult(score=62.5, results=<list of 8 results>),
 EvaluationResult(score=62.5, results=<list of 8 results>))

In [21]:
dspy.inspect_history(n=1)

##No examples





[2026-03-22T09:37:52.908395]

System message:

Your input fields are:
1. `resume` (str): A brief description of candidate's resume
Your output fields are:
1. `reasoning` (str): 
2. `role_fit` (Literal['strong', 'moderate', 'weak']): Assess the resume to check     the role fit of the candidate. Ignore any information like pronouns, names, country of origin
3. `experience_level` (Literal['senior', 'mid', 'junior']): Categorize the candidate into one of these buckets. Consider both academic and professional experience
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## resume ## ]]
{resume}

[[ ## reasoning ## ]]
{reasoning}

[[ ## role_fit ## ]]
{role_fit}        # note: the value you produce must exactly match (no extra characters) one of: strong; moderate; weak

[[ ## experience_level ## ]]
{experience_level}        # note: the value you produce must exactly match (no extra characters) one of: senior; mid; junior

[[ ## completed 

## Optimizers 

Note that this is akin to hp tuning in traditional ML

A few key algorithms 





### BootstrapFewShot

dspy.BootstrapFewShot? - The main idea here is that the algo will identify the examples that are rightly classified based on the metric_threshold and accumulate right examples called as max_bootstrapped_demos - which will be added as few-shot examples

Key points:-

a) teacher_settings -> This is if you want to use an expensive/smarter model to "generate" examples to use in prompting while the program runs on a cheaper/less expensive model - A form of knowledge distillation

b) The circular issue - If only right examples are shown and we are optimizng for that what will happen to wrong examples. However, I believe this is a naive appraoch trying to memorize patterns hoping that it will generalize well

c) max_bootstrapped_demos -> generated by model i.e. teacher whereas max_labeled_demos is examples to show from the data that was provided

d) Its important to know that the key thing that changes in the system prompt are only the examples. The actual wording of the prompt will not change in this optimization method.

In [22]:
#Will skip the teacher model for now

##just ignore eval_set naming :)
trainset = screening_evalset[0:2]
testset = screening_evalset[2:]

optimizer_bootstrap = dspy.BootstrapFewShot(
    metric=screen_evaluator,
    #metric_threshold = True,
    max_bootstrapped_demos=4,  
    max_labeled_demos=2,
    max_rounds=10,
    teacher_settings=None
)

optimized_screener = optimizer_bootstrap.compile(
    student=cot_screener,
    trainset=trainset,
#    valset=testset
)


screen_eval = dspy.Evaluate(
    devset = testset, metric=screen_evaluator, display_progress=True
)

screen_eval(cot_screener), screen_eval(optimized_screener)
# The results from optimized screener seems slightly better. It got one more extra

100%|████████████████████████████████████████████████████████| 2/2 [00:00<00:00, 22.38it/s]


Bootstrapped 2 full traces after 1 examples for up to 10 rounds, amounting to 2 attempts.
Average Metric: 3.00 / 6 (50.0%): 100%|█████████████████████| 6/6 [00:00<00:00, 809.58it/s]

2026/03/22 09:37:53 INFO dspy.evaluate.evaluate: Average Metric: 3 / 6 (50.0%)



Average Metric: 4.00 / 6 (66.7%): 100%|█████████████████████| 6/6 [00:00<00:00, 277.72it/s]

2026/03/22 09:37:53 INFO dspy.evaluate.evaluate: Average Metric: 4 / 6 (66.7%)


(EvaluationResult(score=50.0, results=<list of 6 results>),
 EvaluationResult(score=66.67, results=<list of 6 results>))

In [23]:
dspy.inspect_history(n=1)

##WIth bootstrapfewshot - system prompt remains unchanges; See how it added user and
#assistant message





[2026-03-22T09:37:53.094945]

System message:

Your input fields are:
1. `resume` (str): A brief description of candidate's resume
Your output fields are:
1. `reasoning` (str): 
2. `role_fit` (Literal['strong', 'moderate', 'weak']): Assess the resume to check     the role fit of the candidate. Ignore any information like pronouns, names, country of origin
3. `experience_level` (Literal['senior', 'mid', 'junior']): Categorize the candidate into one of these buckets. Consider both academic and professional experience
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## resume ## ]]
{resume}

[[ ## reasoning ## ]]
{reasoning}

[[ ## role_fit ## ]]
{role_fit}        # note: the value you produce must exactly match (no extra characters) one of: strong; moderate; weak

[[ ## experience_level ## ]]
{experience_level}        # note: the value you produce must exactly match (no extra characters) one of: senior; mid; junior

[[ ## completed 

### MIPRO v2 - Multiprompt Instruction Proposal Optimizer Version 2

1) Less naive compared to Bootstrap method. Optimizes both the instructions and the few shot examples for in-context learning.
2) Basically works by creating both few-shot example and new instructions for each predictor in LM program and then searching over it using Bayesian optimization - Just like hp tuining where you would maximize for accuracy - Note that pretty much all optimization techniques in DSPy is maximizing a metric the search here is more discrete in nature as opposed to traidtional optimizatipn that you may be used to in AI
3) Check this out - https://dspy.ai/api/optimizers/MIPROv2/

In [24]:

customers = [
    {"behavior": "Logged in 2x in last 90 days, downgraded plan last month, opened 3 support tickets about billing", "churn_risk": "high", "recommended_action": "offer_discount"},
    {"behavior": "Daily active user, upgraded plan 2 weeks ago, invited 4 team members", "churn_risk": "low", "recommended_action": "no_action"},
    {"behavior": "No login in 45 days, payment failed once, no support tickets", "churn_risk": "high", "recommended_action": "reach_out"},
    {"behavior": "Weekly usage, consistent for 6 months, renewed annual plan", "churn_risk": "low", "recommended_action": "no_action"},
    {"behavior": "Usage dropped 60% last month, removed 2 integrations, browsed cancellation page", "churn_risk": "high", "recommended_action": "reach_out"},
    {"behavior": "Active daily, but opened 5 support tickets in 2 weeks about performance issues", "churn_risk": "medium", "recommended_action": "reach_out"},
    {"behavior": "Moderate usage, on free trial expiring in 3 days, explored pricing page twice", "churn_risk": "medium", "recommended_action": "offer_discount"},
    {"behavior": "Power user, created custom workflows, but competitor announced lower pricing", "churn_risk": "medium", "recommended_action": "offer_discount"},
    {"behavior": "Logged in once to export all data, no other activity in 30 days", "churn_risk": "high", "recommended_action": "reach_out"},
    {"behavior": "Steady usage, no complaints, auto-renewal on, 2 year customer", "churn_risk": "low", "recommended_action": "no_action"},
    {"behavior": "New signup 1 week ago, completed onboarding, connected 2 integrations", "churn_risk": "low", "recommended_action": "no_action"},
    {"behavior": "Usage flat but asked sales about enterprise features and volume pricing", "churn_risk": "low", "recommended_action": "reach_out"},
]



dspy.configure(lm=dspy.LM("anthropic/claude-opus-4-6",api_key=ANTHROPIC_KEY))

#Create train/test set
full_data = [dspy.Example(cust_behaviour).with_inputs("behavior") for cust_behaviour in customers]
train_set = full_data[0:8]
test_set = full_data[8:]


#Create signature
class CustomerChurn(dspy.Signature):
    """ analyze a brief description of the customer behaviour and categorize it into 2 buckets
      Churn risk and recommended action respectively. Keep in kind that its more profitable to 
      retain the existing client than acquiring a new client. But at the same time if the churn
      risk is not high enough do not give away handouts like there is no tomorrow
    """

    behavior:str = dspy.InputField(desc="A one line summary of a call interaction between customer and support team")
    churn_risk:Literal["high", "medium", "low"] = dspy.OutputField(desc="Risk of loosing customer. This is very critical as it impacts the next output")
    recommended_action:Literal["offer_discount", "no_action", "reach_out"] = dspy.OutputField(desc="Consider the churn risk when suggesting a recommended action") 

In [25]:
##Create an metric function

def evaluate_churn(true, pred, trace=None):

    return (true.churn_risk.lower() == pred.churn_risk.lower()) and (
        true.recommended_action.lower() == pred.recommended_action.lower()
    )

churn_evaluator = dspy.Evaluate(
    devset=test_set,
    metric=evaluate_churn,
    provide_traceback=True,
    display_progress=True
    
)

In [26]:
unoptimized_churn_classifier = dspy.ChainOfThought(signature=CustomerChurn)
churn_evaluator(unoptimized_churn_classifier)

Average Metric: 2.00 / 4 (50.0%): 100%|██████████████████████| 4/4 [00:08<00:00,  2.10s/it]

2026/03/22 09:38:01 INFO dspy.evaluate.evaluate: Average Metric: 2 / 4 (50.0%)


EvaluationResult(score=50.0, results=<list of 4 results>)

In [27]:
dspy.inspect_history(n=1)





[2026-03-22T09:38:01.545246]

System message:

Your input fields are:
1. `behavior` (str): A one line summary of a call interaction between customer and support team
Your output fields are:
1. `reasoning` (str): 
2. `churn_risk` (Literal['high', 'medium', 'low']): Risk of loosing customer. This is very critical as it impacts the next output
3. `recommended_action` (Literal['offer_discount', 'no_action', 'reach_out']): Consider the churn risk when suggesting a recommended action
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## behavior ## ]]
{behavior}

[[ ## reasoning ## ]]
{reasoning}

[[ ## churn_risk ## ]]
{churn_risk}        # note: the value you produce must exactly match (no extra characters) one of: high; medium; low

[[ ## recommended_action ## ]]
{recommended_action}        # note: the value you produce must exactly match (no extra characters) one of: offer_discount; no_action; reach_out

[[ ## completed ## ]]
In adher

In [38]:
## Running MIPROv2 optimizer - run it just like bootsratp

mipro = dspy.MIPROv2(
    metric = evaluate_churn,
    max_bootstrapped_demos = 6,
    max_labeled_demos = 6,
    auto="medium",
    seed=8,
    init_temperature=0.0,
    verbose=True
)

optimized_mipro = mipro.compile(
    student=unoptimized_churn_classifier,
    trainset=train_set)

2026/03/22 09:48:01 INFO dspy.teleprompt.mipro_optimizer_v2: 
RUNNING WITH THE FOLLOWING MEDIUM AUTO RUN SETTINGS:
num_trials: 18
minibatch: False
num_fewshot_candidates: 12
num_instruct_candidates: 6
valset size: 6

2026/03/22 09:48:01 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 1: BOOTSTRAP FEWSHOT EXAMPLES <==
2026/03/22 09:48:01 INFO dspy.teleprompt.mipro_optimizer_v2: These will be used as few-shot example candidates for our program and for creating instructions.

2026/03/22 09:48:01 INFO dspy.teleprompt.mipro_optimizer_v2: Bootstrapping N=12 sets of demonstrations...


Bootstrapping set 1/12
Bootstrapping set 2/12
Bootstrapping set 3/12


100%|████████████████████████████████████████████████████████| 2/2 [00:00<00:00, 15.17it/s]


Bootstrapped 2 full traces after 1 examples for up to 1 rounds, amounting to 2 attempts.
Bootstrapping set 4/12


100%|████████████████████████████████████████████████████████| 2/2 [00:00<00:00, 23.29it/s]


...

2026/03/22 09:48:04 INFO dspy.evaluate.evaluate: Average Metric: 2 / 6 (33.3%)
2026/03/22 09:48:04 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 33.33 with parameters ['Predictor 0: Instruction 0', 'Predictor 0: Few-Shot Set 7'].
2026/03/22 09:48:04 INFO dspy.teleprompt.mipro_optimizer_v2: Scores so far: [50.0, 33.33, 33.33, 33.33, 33.33, 33.33, 33.33, 50.0, 33.33, 33.33, 33.33, 50.0, 50.0, 33.33, 33.33, 33.33, 33.33]
2026/03/22 09:48:04 INFO dspy.teleprompt.mipro_optimizer_v2: Best score so far: 50.0
2026/03/22 09:48:04 INFO dspy.teleprompt.mipro_optimizer_v2: =========================


2026/03/22 09:48:04 INFO dspy.teleprompt.mipro_optimizer_v2: ===== Trial 18 / 18 =====
2026/03/22 09:48:04 INFO dspy.teleprompt.mipro_optimizer_v2: Evaluating the following candidate program...




Predictor 0
i: analyze a brief description of the customer behaviour and categorize it into 2 buckets
Churn risk and recommended action respectively. Keep in kind that its more profitable to 
retain the existing client than acquiring a new client. But at the same time if the churn
risk is not high enough do not give away handouts like there is no tomorrow
p: Recommended Action:


Average Metric: 3.00 / 6 (50.0%): 100%|████████████████████| 6/6 [00:00<00:00, 1113.68it/s]

2026/03/22 09:48:04 INFO dspy.evaluate.evaluate: Average Metric: 3 / 6 (50.0%)
2026/03/22 09:48:04 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 50.0 with parameters ['Predictor 0: Instruction 0', 'Predictor 0: Few-Shot Set 9'].
2026/03/22 09:48:04 INFO dspy.teleprompt.mipro_optimizer_v2: Scores so far: [50.0, 33.33, 33.33, 33.33, 33.33, 33.33, 33.33, 50.0, 33.33, 33.33, 33.33, 50.0, 50.0, 33.33, 33.33, 33.33, 33.33, 50.0]
2026/03/22 09:48:04 INFO dspy.teleprompt.mipro_optimizer_v2: Best score so far: 50.0
2026/03/22 09:48:04 INFO dspy.teleprompt.mipro_optimizer_v2: =========================


2026/03/22 09:48:04 INFO dspy.teleprompt.mipro_optimizer_v2: ===== Trial 19 / 18 =====
2026/03/22 09:48:04 INFO dspy.teleprompt.mipro_optimizer_v2: Evaluating the following candidate program...




Predictor 0
i: You are an expert customer retention analyst. Given a brief description of a customer's behavior or support interaction, analyze the signals step by step and produce a churn risk classification and a recommended retention action.

**How to analyze:**
1. Carefully evaluate all behavioral signals in the description, including: login/usage frequency, plan changes (upgrades vs. downgrades), support ticket volume and topics (especially billing complaints), team adoption and expansion, and overall engagement patterns.
2. Weigh these signals together to determine the overall churn risk:
   - **high**: Customer shows multiple disengagement signals such as very low login frequency, plan downgrades, billing complaints, or explicit cancellation mentions. These customers are at serious risk of leaving.
   - **medium**: Customer shows mixed signals — some engagement but also some concerning behaviors like reduced usage, minor complaints, or stagnation without growth. They aren't act

2026/03/22 09:48:04 INFO dspy.evaluate.evaluate: Average Metric: 2 / 6 (33.3%)
2026/03/22 09:48:04 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 33.33 with parameters ['Predictor 0: Instruction 4', 'Predictor 0: Few-Shot Set 0'].
2026/03/22 09:48:04 INFO dspy.teleprompt.mipro_optimizer_v2: Scores so far: [50.0, 33.33, 33.33, 33.33, 33.33, 33.33, 33.33, 50.0, 33.33, 33.33, 33.33, 50.0, 50.0, 33.33, 33.33, 33.33, 33.33, 50.0, 33.33]
2026/03/22 09:48:04 INFO dspy.teleprompt.mipro_optimizer_v2: Best score so far: 50.0
2026/03/22 09:48:04 INFO dspy.teleprompt.mipro_optimizer_v2: =========================


2026/03/22 09:48:04 INFO dspy.teleprompt.mipro_optimizer_v2: Returning best identified program with score 50.0!


In [39]:
churn_evaluator(optimized_mipro)
# The results didnt change

Average Metric: 2.00 / 4 (50.0%): 100%|█████████████████████| 4/4 [00:00<00:00, 519.00it/s]

2026/03/22 09:48:04 INFO dspy.evaluate.evaluate: Average Metric: 2 / 4 (50.0%)


EvaluationResult(score=50.0, results=<list of 4 results>)